In [3]:
# =========================================================
# 1. IMPORTS Y CARGA DE DATOS
# =========================================================

import pandas as pd
import numpy as np
import time
import os

# Gráficos
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# --- scikit-learn ---
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Modelos
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


In [4]:
df = pd.read_excel("../Datos/Limpios/información_préstamos_limpio.xlsx")

# Eliminamos filas sin Prima (target)
df = df.dropna(subset=["Prima"]).copy()

In [5]:
## target y variables 

target = "Prima"

# Columnas que NO queremos usar como predictoras
cols_to_exclude = ["ID", "Prima", "Impago"]

feature_cols = [c for c in df.columns if c not in cols_to_exclude]

X = df[feature_cols]
y = df[target]

# Detectamos variables numéricas y categóricas
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Por seguridad, filtramos solo las que siguen en X
numeric_features = [c for c in numeric_features if c in X.columns]
categorical_features = [c for c in categorical_features if c in X.columns]

In [6]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [7]:
# =========================================================
# 3. PREPROCESAMIENTO
# =========================================================

# Preprocesamiento GENERAL (sin escalado) -> árboles, RF, XGBoost, LightGBM
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Preprocesamiento para regresión lineal (con ESCALADO)
numeric_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_features),
        ("cat", categorical_transformer_lr, categorical_features),
    ]
)

In [8]:
# =========================================================
# 4. FUNCIÓN DE MÉTRICAS
# =========================================================

def calcular_metricas(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)

    # Evita división por 0 por seguridad
    eps = 1e-8
    mape = np.mean(
        np.abs((y_true_arr - y_pred_arr) / np.maximum(np.abs(y_true_arr), eps))
    ) * 100

    return rmse, mae, r2, mape


In [9]:
# =========================================================
# 5. MODELO XGBOOST
# =========================================================

xgb_reg = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_estimators=200,
    n_jobs=-1
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", xgb_reg),
    ]
)

param_grid_xgb = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__gamma": [0, 0.1, 0.5],
    "model__reg_lambda": [1, 5, 10],
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid_xgb,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando XGBoost con GridSearchCV...")

t0 = time.time()
xgb_grid.fit(X_train, y_train)
train_time_xgb = time.time() - t0

print("\nMejores hiperparámetros encontrados para XGBoost:")
print(xgb_grid.best_params_)

best_xgb = xgb_grid.best_estimator_

t0 = time.time()
y_pred_xgb = best_xgb.predict(X_test)
pred_time_xgb = time.time() - t0

rmse_xgb, mae_xgb, r2_xgb, mape_xgb = calcular_metricas(y_test, y_pred_xgb)
r2_train_xgb = best_xgb.score(X_train, y_train)
overfitting_gap_xgb = r2_train_xgb - r2_xgb

print("\nResultados XGBoost en test:")
print(f"  RMSE: {rmse_xgb:.4f}")
print(f"  MAE : {mae_xgb:.4f}")
print(f"  R²  : {r2_xgb:.4f}")
print(f"  MAPE: {mape_xgb:.2f}%")
print(f"  Tiempo entrenamiento (s): {train_time_xgb:.3f}")
print(f"  Tiempo predicción   (s): {pred_time_xgb:.6f}")
print(f"  Overfitting gap (R² train - test): {overfitting_gap_xgb:.4f}")

Entrenando XGBoost con GridSearchCV...
Fitting 5 folds for each of 1458 candidates, totalling 7290 fits

Mejores hiperparámetros encontrados para XGBoost:
{'model__colsample_bytree': 1.0, 'model__gamma': 0, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 400, 'model__reg_lambda': 1, 'model__subsample': 0.6}

Resultados XGBoost en test:
  RMSE: 25.9236
  MAE : 20.7565
  R²  : 0.9683
  MAPE: 3.74%
  Tiempo entrenamiento (s): 921.219
  Tiempo predicción   (s): 0.015970
  Overfitting gap (R² train - test): 0.0076


In [10]:
# =========================================================
# 6. RANDOM FOREST
# =========================================================

rf_reg = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", rf_reg),
    ]
)

param_grid_rf = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10],
    "model__max_features": ["sqrt", "log2"],
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid_rf,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("\nEntrenando Random Forest con GridSearchCV...")

t0 = time.time()
rf_grid.fit(X_train, y_train)
train_time_rf = time.time() - t0

print("\nMejores hiperparámetros encontrados para Random Forest:")
print(rf_grid.best_params_)

best_rf = rf_grid.best_estimator_

t0 = time.time()
y_pred_rf = best_rf.predict(X_test)
pred_time_rf = time.time() - t0

rmse_rf, mae_rf, r2_rf, mape_rf = calcular_metricas(y_test, y_pred_rf)
r2_train_rf = best_rf.score(X_train, y_train)
overfitting_gap_rf = r2_train_rf - r2_rf

print("\nResultados Random Forest en test:")
print(f"  RMSE: {rmse_rf:.4f}")
print(f"  MAE : {mae_rf:.4f}")
print(f"  R²  : {r2_rf:.4f}")
print(f"  MAPE: {mape_rf:.2f}%")
print(f"  Tiempo entrenamiento (s): {train_time_rf:.3f}")
print(f"  Tiempo predicción   (s): {pred_time_rf:.6f}")
print(f"  Overfitting gap (R² train - test): {overfitting_gap_rf:.4f}")


Entrenando Random Forest con GridSearchCV...
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Mejores hiperparámetros encontrados para Random Forest:
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__n_estimators': 300}

Resultados Random Forest en test:
  RMSE: 68.3919
  MAE : 56.3573
  R²  : 0.7796
  MAPE: 10.68%
  Tiempo entrenamiento (s): 19.949
  Tiempo predicción   (s): 0.155128
  Overfitting gap (R² train - test): 0.1866


In [16]:
# =========================================================
# RIDGE (Regresión lineal con regularización) + métricas + tiempos + gap
# =========================================================

import time
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

# Preprocesado para regresión lineal (con escalado)
numeric_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_features),
        ("cat", categorical_transformer_lr, categorical_features),
    ]
)

# Modelo Ridge
ridge_reg = Ridge()

ridge_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_lr),
        ("model", ridge_reg),
    ]
)

# Grid de hiperparámetros
param_grid_ridge = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 50.0]
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_ridge,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando Ridge con GridSearchCV...")

t0 = time.time()
ridge_grid.fit(X_train, y_train)
train_time_ridge = time.time() - t0

print("\nMejores hiperparámetros para Ridge:")
print(ridge_grid.best_params_)

best_ridge = ridge_grid.best_estimator_

# Predicción + tiempos
t0 = time.time()
y_pred_ridge = best_ridge.predict(X_test)
pred_time_ridge = time.time() - t0

# Métricas
rmse_ridge, mae_ridge, r2_ridge, mape_ridge = calcular_metricas(y_test, y_pred_ridge)

# Gap sobreajuste
r2_train_ridge = best_ridge.score(X_train, y_train)
overfitting_gap_ridge = r2_train_ridge - r2_ridge

print("\nResultados Ridge en test:")
print(f"  RMSE: {rmse_ridge:.4f}")
print(f"  MAE : {mae_ridge:.4f}")
print(f"  R²  : {r2_ridge:.4f}")
print(f"  MAPE: {mape_ridge:.2f}%")
print(f"  Tiempo entrenamiento (s): {train_time_ridge:.3f}")
print(f"  Tiempo predicción   (s): {pred_time_ridge:.6f}")
print(f"  Overfitting gap (R² train - test): {overfitting_gap_ridge:.4f}")

Entrenando Ridge con GridSearchCV...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Mejores hiperparámetros para Ridge:
{'model__alpha': 1.0}

Resultados Ridge en test:
  RMSE: 73.0399
  MAE : 58.9141
  R²  : 0.7486
  MAPE: 10.61%
  Tiempo entrenamiento (s): 8.161
  Tiempo predicción   (s): 0.004505
  Overfitting gap (R² train - test): 0.0044


In [11]:
# =========================================================
# 7. ÁRBOL DE REGRESIÓN (DECISION TREE)
# =========================================================

tree_reg = DecisionTreeRegressor(
    random_state=42
)

tree_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", tree_reg),
    ]
)

param_grid_tree = {
    "model__max_depth": [None, 3, 5, 7, 10],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
}

tree_grid = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid_tree,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("\nEntrenando Árbol de Regresión con GridSearchCV...")

t0 = time.time()
tree_grid.fit(X_train, y_train)
train_time_tree = time.time() - t0

print("\nMejores hiperparámetros encontrados para el Árbol de Regresión:")
print(tree_grid.best_params_)

best_tree = tree_grid.best_estimator_

t0 = time.time()
y_pred_tree = best_tree.predict(X_test)
pred_time_tree = time.time() - t0

rmse_tree, mae_tree, r2_tree, mape_tree = calcular_metricas(y_test, y_pred_tree)
r2_train_tree = best_tree.score(X_train, y_train)
overfitting_gap_tree = r2_train_tree - r2_tree

print("\nResultados Árbol de Regresión en test:")
print(f"  RMSE: {rmse_tree:.4f}")
print(f"  MAE : {mae_tree:.4f}")
print(f"  R²  : {r2_tree:.4f}")
print(f"  MAPE: {mape_tree:.2f}%")
print(f"  Tiempo entrenamiento (s): {train_time_tree:.3f}")
print(f"  Tiempo predicción   (s): {pred_time_tree:.6f}")
print(f"  Overfitting gap (R² train - test): {overfitting_gap_tree:.4f}")


Entrenando Árbol de Regresión con GridSearchCV...
Fitting 5 folds for each of 80 candidates, totalling 400 fits

Mejores hiperparámetros encontrados para el Árbol de Regresión:
{'model__max_depth': None, 'model__min_samples_leaf': 8, 'model__min_samples_split': 20}

Resultados Árbol de Regresión en test:
  RMSE: 66.5120
  MAE : 51.7547
  R²  : 0.7916
  MAPE: 9.28%
  Tiempo entrenamiento (s): 3.552
  Tiempo predicción   (s): 0.007997
  Overfitting gap (R² train - test): 0.1022


In [12]:
# =========================================================
# 8. REGRESIÓN LINEAL (SIN REGULARIZACIÓN)
# =========================================================

lin_reg = LinearRegression()

lin_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_lr),
        ("model", lin_reg),
    ]
)

print("\nEntrenando Regresión Lineal (sin regularización)...")

t0 = time.time()
lin_pipeline.fit(X_train, y_train)
train_time_lin = time.time() - t0

t0 = time.time()
y_pred_lin = lin_pipeline.predict(X_test)
pred_time_lin = time.time() - t0

rmse_lin, mae_lin, r2_lin, mape_lin = calcular_metricas(y_test, y_pred_lin)
r2_train_lin = lin_pipeline.score(X_train, y_train)
overfitting_gap_lin = r2_train_lin - r2_lin

print("\nResultados Regresión Lineal (sin regularización) en test:")
print(f"  RMSE: {rmse_lin:.4f}")
print(f"  MAE : {mae_lin:.4f}")
print(f"  R²  : {r2_lin:.4f}")
print(f"  MAPE: {mape_lin:.2f}%")
print(f"  Tiempo entrenamiento (s): {train_time_lin:.3f}")
print(f"  Tiempo predicción   (s): {pred_time_lin:.6f}")
print(f"  Overfitting gap (R² train - test): {overfitting_gap_lin:.4f}")


Entrenando Regresión Lineal (sin regularización)...

Resultados Regresión Lineal (sin regularización) en test:
  RMSE: 73.0405
  MAE : 58.9104
  R²  : 0.7486
  MAPE: 10.61%
  Tiempo entrenamiento (s): 0.137
  Tiempo predicción   (s): 0.010996
  Overfitting gap (R² train - test): 0.0044


In [13]:
# =========================================================
# 9. LASSO
# =========================================================

lasso_reg = Lasso(
    max_iter=10000,
    random_state=42
)

lasso_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_lr),
        ("model", lasso_reg),
    ]
)

param_grid_lasso = {
    "model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]
}

lasso_grid = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=param_grid_lasso,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("\nEntrenando LASSO con GridSearchCV...")

t0 = time.time()
lasso_grid.fit(X_train, y_train)
train_time_lasso = time.time() - t0

print("\nMejores hiperparámetros para LASSO:")
print(lasso_grid.best_params_)

best_lasso = lasso_grid.best_estimator_

t0 = time.time()
y_pred_lasso = best_lasso.predict(X_test)
pred_time_lasso = time.time() - t0

rmse_lasso, mae_lasso, r2_lasso, mape_lasso = calcular_metricas(y_test, y_pred_lasso)
r2_train_lasso = best_lasso.score(X_train, y_train)
overfitting_gap_lasso = r2_train_lasso - r2_lasso

print("\nResultados LASSO en test:")
print(f"  RMSE: {rmse_lasso:.4f}")
print(f"  MAE : {mae_lasso:.4f}")
print(f"  R²  : {r2_lasso:.4f}")
print(f"  MAPE: {mape_lasso:.2f}%")
print(f"  Tiempo entrenamiento (s): {train_time_lasso:.3f}")
print(f"  Tiempo predicción   (s): {pred_time_lasso:.6f}")
print(f"  Overfitting gap (R² train - test): {overfitting_gap_lasso:.4f}")


Entrenando LASSO con GridSearchCV...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Mejores hiperparámetros para LASSO:
{'model__alpha': 0.01}

Resultados LASSO en test:
  RMSE: 73.0416
  MAE : 58.9104
  R²  : 0.7486
  MAPE: 10.61%
  Tiempo entrenamiento (s): 0.812
  Tiempo predicción   (s): 0.008008
  Overfitting gap (R² train - test): 0.0044


In [14]:
# =========================================================
# 10. LIGHTGBM
# =========================================================

lgbm_reg = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_estimators=200,
    n_jobs=-1
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", lgbm_reg),
    ]
)

param_grid_lgbm = {
    "model__n_estimators": [200, 400],
    "model__num_leaves": [31, 63, 127],
    "model__max_depth": [-1, 5, 10],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__reg_lambda": [0.0, 1.0, 5.0],
}

lgbm_grid = GridSearchCV(
    estimator=lgbm_pipeline,
    param_grid=param_grid_lgbm,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("\nEntrenando LightGBM con GridSearchCV...")

t0 = time.time()
lgbm_grid.fit(X_train, y_train)
train_time_lgbm = time.time() - t0

print("\nMejores hiperparámetros encontrados para LightGBM:")
print(lgbm_grid.best_params_)

best_lgbm = lgbm_grid.best_estimator_

t0 = time.time()
y_pred_lgbm = best_lgbm.predict(X_test)
pred_time_lgbm = time.time() - t0

rmse_lgbm, mae_lgbm, r2_lgbm, mape_lgbm = calcular_metricas(y_test, y_pred_lgbm)
r2_train_lgbm = best_lgbm.score(X_train, y_train)
overfitting_gap_lgbm = r2_train_lgbm - r2_lgbm

print("\nResultados LightGBM en test:")
print(f"  RMSE: {rmse_lgbm:.4f}")
print(f"  MAE : {mae_lgbm:.4f}")
print(f"  R²  : {r2_lgbm:.4f}")
print(f"  MAPE: {mape_lgbm:.2f}%")
print(f"  Tiempo entrenamiento (s): {train_time_lgbm:.3f}")
print(f"  Tiempo predicción   (s): {pred_time_lgbm:.6f}")
print(f"  Overfitting gap (R² train - test): {overfitting_gap_lgbm:.4f}")


Entrenando LightGBM con GridSearchCV...
Fitting 5 folds for each of 1458 candidates, totalling 7290 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000717 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1659
[LightGBM] [Info] Number of data points in the train set: 5311, number of used features: 25
[LightGBM] [Info] Start training from score 590.896198
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

c:\Users\ander\anaconda3\envs\RETO_07_MORADO\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ander\anaconda3\envs\RETO_07_MORADO\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [17]:
# =========================================================
# 11. TABLA DE RESULTADOS (MÉTRICAS + TIEMPOS + GAP)
# =========================================================

resultados = pd.DataFrame({
    "Modelo": [
        "Árbol",
        "RandomForest",
        "XGBoost",
        "LightGBM",
        "Regresión lineal",
        "Ridge",
        "Lasso"
    ],
    "RMSE": [
        rmse_tree, rmse_rf, rmse_xgb, rmse_lgbm,
        rmse_lin, rmse_ridge, rmse_lasso
    ],
    "MAE":  [
        mae_tree, mae_rf, mae_xgb, mae_lgbm,
        mae_lin, mae_ridge, mae_lasso
    ],
    "R2":   [
        r2_tree, r2_rf, r2_xgb, r2_lgbm,
        r2_lin, r2_ridge, r2_lasso
    ],
    "MAPE (%)": [
        mape_tree, mape_rf, mape_xgb, mape_lgbm,
        mape_lin, mape_ridge, mape_lasso
    ],
    "Train_time_sec": [
        train_time_tree, train_time_rf, train_time_xgb,
        train_time_lgbm, train_time_lin, train_time_ridge,
        train_time_lasso
    ],
    "Pred_time_sec": [
        pred_time_tree, pred_time_rf, pred_time_xgb,
        pred_time_lgbm, pred_time_lin, pred_time_ridge,
        pred_time_lasso
    ],
    "Overfitting_gap_R2": [
        overfitting_gap_tree, overfitting_gap_rf, overfitting_gap_xgb,
        overfitting_gap_lgbm, overfitting_gap_lin, overfitting_gap_ridge,
        overfitting_gap_lasso
    ]
})

print("\nTabla de resultados ordenada por RMSE:")
print(resultados.sort_values("RMSE"))


Tabla de resultados ordenada por RMSE:
             Modelo       RMSE        MAE        R2   MAPE (%)  \
2           XGBoost  25.923587  20.756538  0.968336   3.742727   
3          LightGBM  27.148522  20.926362  0.965273   3.663040   
0             Árbol  66.512011  51.754724  0.791563   9.278825   
1      RandomForest  68.391865  56.357260  0.779614  10.683694   
5             Ridge  73.039876  58.914079  0.748641  10.609495   
4  Regresión lineal  73.040494  58.910446  0.748637  10.607050   
6             Lasso  73.041599  58.910440  0.748629  10.608146   

   Train_time_sec  Pred_time_sec  Overfitting_gap_R2  
2      921.218842       0.015970            0.007649  
3     4861.806329       0.028614            0.026244  
0        3.551558       0.007997            0.102212  
1       19.949311       0.155128            0.186623  
5        8.161267       0.004505            0.004424  
4        0.137275       0.010996            0.004429  
6        0.811767       0.008008            0.

In [18]:
# =========================================================
# 12. ESTIMACIÓN DEL COSTE ESPERADO DEL SEGURO (XGBoost GANADOR)
# =========================================================

df_test = X_test.copy()
df_test["Prima_real"] = y_test.values
df_test["Coste_Esperado_Seguro"] = best_xgb.predict(X_test)

print("\nEjemplo df_test con Prima real y coste esperado:")
print(df_test.head())



Ejemplo df_test con Prima real y coste esperado:
      Edad  Ingresos  Monto_Inicial  Scoring_Crediticio  Meses_Empleo  \
1652    20     16992          45593                 711            19   
5302    21     19574          46664                 848            29   
2986    69     37244          40000                 367           473   
1545    34     26596          53155                 787           147   
6016    68     39808          46344                 575           499   

      Num_Creditos  Ratio_Interes  Duracion  Ratio_Deuda_Ingresos  \
1652             3          22.35        60                  0.47   
5302             1          13.80        36                  0.55   
2986             2           3.81        48                  0.26   
1545             3           9.69        24                  0.55   
6016             2          19.15        48                  0.30   

                 Estudios Tipo_Jornada_Laboral Estado_Civil  \
1652              Escolar     jor

In [32]:
# =========================================================
# 13. GRÁFICO REAL vs PREDICHO (PLOTLY) + GUARDADO PNG
# =========================================================

COLORES = {
    'Verde_Fuerte': '#74b404', 
    'Verde_Claro':  '#cdfc7d', 
    'Rojo_Corp':    '#aa044c', 
    'Morado_Os':    '#876784',
    'Morado_Cl':    '#b09eae'
}

y_pred = best_xgb.predict(X_test)

df_plot = pd.DataFrame({
    "Prima_real": y_test.values,
    "Prima_predicha": y_pred
})

fig = px.scatter(
    df_plot,
    x="Prima_real",
    y="Prima_predicha",
    opacity=0.5,
    color_discrete_sequence=[COLORES["Verde_Fuerte"]],
    labels={
        "Prima_real": "Prima Real",
        "Prima_predicha": "Prima Predicha"
    },
    title="Real vs Predicho - XGBoost"
)

# Línea perfecta (y = x)
min_val = min(df_plot.min())
max_val = max(df_plot.max())

fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(
            color=COLORES["Rojo_Corp"],
            dash="dash",
            width=3
        ),
        name="Predicción Perfecta"
    )
)

fig.update_layout(template="plotly_white")

fig.show()

# Carpeta de salida para los gráficos
CARPETA_SALIDA = "Graficos_Informe"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

def guardar_grafico(fig, nombre_archivo):
    ruta = os.path.join(CARPETA_SALIDA, f"{nombre_archivo}.png")
    fig.write_image(ruta, width=1200, height=800, scale=2)
    print(f"Gráfico guardado en: {ruta}")

guardar_grafico(fig, "real_vs_predicho_xgboost")

Gráfico guardado en: Graficos_Informe\real_vs_predicho_xgboost.png


In [34]:
# =========================================================
# 14. IMPORTANCIA DE VARIABLES (XGBOOST) - PLOTLY
# =========================================================

import os
import pandas as pd
import plotly.express as px

COLORES = {
    'Verde_Fuerte': '#74b404', 
    'Verde_Claro':  '#cdfc7d', 
    'Rojo_Corp':    '#aa044c', 
    'Morado_Os':    '#876784',
    'Morado_Cl':    '#b09eae'
}

CARPETA_SALIDA = "graficos"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

def guardar_grafico(fig, nombre_archivo):
    ruta = os.path.join(CARPETA_SALIDA, f"{nombre_archivo}.png")
    fig.write_image(ruta, width=1200, height=800, scale=2)
    print(f"Gráfico guardado en: {ruta}")

# 1) Importancias y nombres de variables del pipeline
importancias = best_xgb.named_steps["model"].feature_importances_
feature_names = best_xgb.named_steps["preprocess"].get_feature_names_out()

df_importancia = (
    pd.DataFrame({"Variable": feature_names, "Importancia": importancias})
    .sort_values("Importancia", ascending=False)
    .head(15)
    .sort_values("Importancia")  # para que en horizontal quede "de menor a mayor" y se lea bien
)

# 2) Gráfico con Plotly
fig_imp = px.bar(
    df_importancia,
    x="Importancia",
    y="Variable",
    orientation="h",
    title="Top 15 Variables Más Importantes - XGBoost",
    color_discrete_sequence=[COLORES["Verde_Fuerte"]],
    labels={"Importancia": "Importancia", "Variable": "Variable"}
)

fig_imp.update_layout(template="plotly_white")

# 3) Mostrar y guardar
fig_imp.show()
guardar_grafico(fig_imp, "importancia_variables_xgboost")

Gráfico guardado en: graficos\importancia_variables_xgboost.png


In [30]:
# =========================================================
# 15. VALIDACIÓN CRUZADA (R²) PARA XGBOOST GANADOR
# =========================================================

scores = cross_val_score(best_xgb, X, y, cv=5, scoring="r2")
print("\nR2 CV (XGBoost):", scores)
print("R2 CV media:", scores.mean())


R2 CV (XGBoost): [0.96074012 0.96076393 0.96161706 0.96336915 0.90216376]
R2 CV media: 0.9497308075805831
